# Demo 2: Controlled Pendulum
## Demo 2.3: FMPy Co-Simulation

### Description

The following demo implements a controlled pendulum system described in Demo 2.1. The Modelica models for the `Reference`, `Pendulum`, `AngleEncoder`, `Controller`, and `Drive` are exported as an Co-Simulation FMU using OpenModelica. The FMUs are then imported and simulated in Python using the FMPy package.

OpenModelica allows  the FMI 3.0 standard for Co-Simulation FMUs, which is supported by FMPy.
However, OMPython currently only supports exporting FMUs in the FMI 2.0 standard. Therefore, the FMUs are exported in the FMI 2.0 standard and simulated using FMPy.

### Procedure

**1. Changing the working dirctory to use `SysSimX` package**

In [2]:
import os
import sys
from pathlib import Path
repo_root = Path.cwd().parent.parent
sys.path.insert(0, str(repo_root))

from SysSimX.utilities.update_fmus import get_fmu_paths, get_models_within_package

**2. Get the FMU files**

In [3]:
demo_dir_path = Path(repo_root / 'demos' / 'ControlledPendulum/MyModels/Modelica')
package_path = Path(demo_dir_path / 'ControlledPendulum')
fmu_output_dir = Path(demo_dir_path / 'MyModels/FMUs')

fmu_paths = get_fmu_paths(package_path, fmu_output_dir, force_rebuild=False)
for fmu, path in fmu_paths.items():
    print(f"{fmu:<25}: {path}")

All FMUs for package 'ControlledPendulum' already exist. Skipping generation.
AngleEncoder             : /home/flo/repos/SystemSimulation/demos/ControlledPendulum/MyModels/Modelica/MyModels/FMUs/AngleEncoder.fmu
Demo_Driven              : /home/flo/repos/SystemSimulation/demos/ControlledPendulum/MyModels/Modelica/MyModels/FMUs/Demo_Driven.fmu
Drive                    : /home/flo/repos/SystemSimulation/demos/ControlledPendulum/MyModels/Modelica/MyModels/FMUs/Drive.fmu
PID_Continuous           : /home/flo/repos/SystemSimulation/demos/ControlledPendulum/MyModels/Modelica/MyModels/FMUs/PID_Continuous.fmu
Pendulum                 : /home/flo/repos/SystemSimulation/demos/ControlledPendulum/MyModels/Modelica/MyModels/FMUs/Pendulum.fmu
Reference                : /home/flo/repos/SystemSimulation/demos/ControlledPendulum/MyModels/Modelica/MyModels/FMUs/Reference.fmu


**3. Load the Model Descriptions and Setup Variables**

- FMUs provide a model description file in XML format that contains information about the model structure, variables, and capabilities.
- The model description is parsed when loading the FMU, and variable information can be accessed using getter methods.
- Each variable has a unique value reference (VR) used for setting and getting variable values.
- The variable references are stored in dictionaries for easy access during the simulation loop.

In [4]:
# Import FMPy
from fmpy import read_model_description

fmu_dict = {}

# Read model descriptions and variable references
for model_name, s in fmu_paths.items():
    fmu_dict[model_name] = {}
    fmu_dict[model_name]["ModelDescription"] = read_model_description(s)
    vrs = {}
    for variable in fmu_dict[model_name]["ModelDescription"].modelVariables:
        vrs[variable.name] = variable.valueReference
    fmu_dict[model_name]["ValueRefs"] = vrs

In [5]:
from fmpy import read_model_description

# Read Model Description
path = fmu_paths['Pendulum']
md = read_model_description(path)
variables = md.modelVariables
model_variables = {}
for variable in variables:
    temp_dict = {}
    temp_dict['name'] = variable.name
    temp_dict['unit'] = variable.unit
    temp_dict['ValueReference'] = variable.valueReference
    temp_dict['causality'] = variable.causality
    temp_dict['variability'] = variable.variability
    model_variables[variable.name] = temp_dict

# Print header
print(f"FMU: {path}")
print(90 * "-")
print(f"{'Variable Name':<31} : {'ValueRef':<10} : {'Causality':<10} : {'Variability':<10} : {'Unit'}")
print(90 * "-")
# Print variable details
for name, var in model_variables.items():
    print(f"{name:<31} : {var['ValueReference']:<10} : {var['causality']:<10} : {var['variability']:<10} : {var['unit']}")


FMU: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/MyModels/Modelica/MyModels/FMUs/Pendulum.fmu
------------------------------------------------------------------------------------------
Variable Name                   : ValueRef   : Causality  : Variability : Unit
------------------------------------------------------------------------------------------
_D_outputAlias_omega_state      : 0          : local      : continuous : rad/s
_D_outputAlias_q_state          : 1          : local      : continuous : rad
der(_D_outputAlias_omega_state) : 2          : local      : continuous : s-2
der(_D_outputAlias_q_state)     : 3          : local      : continuous : Hz
omega_state                     : 6          : output     : continuous : rad/s
q_state                         : 7          : output     : continuous : rad
torque                          : 8          : input      : continuous : N.m
L                               : 9          : parameter  : fixed      : m
m             

In [6]:
from fmpy import extract
from fmpy.fmi2 import FMU2Slave

# Instantiate Pendulum
guid = fmu_dict['Pendulum']["ModelDescription"].guid
unzip_dir = extract(fmu_paths['Pendulum'])
modelIdentifier = md.coSimulation.modelIdentifier
pendulum_fmu = FMU2Slave(guid=guid,
                          unzipDirectory=unzip_dir,
                          modelIdentifier=modelIdentifier,
                          instanceName='pendulum_fmu')
pendulum_fmu.instantiate()

LOG_SOLVER        | info    | CVODE linear multistep method CV_BDF
LOG_SOLVER        | info    | CVODE maximum integration order CV_ITER_NEWTON
LOG_SOLVER        | info    | CVODE use equidistant time grid YES
LOG_SOLVER        | info    | CVODE Using relative error tolerance 1.000000e-06
LOG_SOLVER        | info    | CVODE Using dense internal linear solver SUNLinSol_Dense.
LOG_SOLVER        | info    | CVODE Use internal dense numeric jacobian method.
LOG_SOLVER        | info    | CVODE uses internal root finding method NO
LOG_SOLVER        | info    | CVODE maximum absolut step size 0
LOG_SOLVER        | info    | CVODE initial step size is set automatically
LOG_SOLVER        | info    | CVODE maximum integration order 5
LOG_SOLVER        | info    | CVODE maximum number of nonlinear convergence failures permitted during one step 10
LOG_SOLVER        | info    | CVODE BDF stability limit detection algorithm OFF


In [7]:
# Dictionary to hold references to parameters
params_rf = {}

params_rf['ref_mean'] = fmu_dict['Reference']['ValueRefs']['mean']
params_rf['ref_amplitude'] = fmu_dict['Reference']['ValueRefs']['amplitude']
params_rf['ref_frequency'] = fmu_dict['Reference']['ValueRefs']['frequency']

params_rf['pendulum_q0'] = fmu_dict['Pendulum']['ValueRefs']['q0']

In [8]:
# Dictionary to hold variable references
vars_rf = {}
vars_rf['q_ref'] = fmu_dict['Reference']['ValueRefs']['q_ref']

vars_rf['q_state'] = fmu_dict['Pendulum']['ValueRefs']['q_state']
vars_rf['omega_state'] = fmu_dict['Pendulum']['ValueRefs']['omega_state']
vars_rf['torque'] = fmu_dict['Pendulum']['ValueRefs']['torque']

vars_rf['q_sensor'] = fmu_dict['AngleEncoder']['ValueRefs']['q']
vars_rf['U_q'] = fmu_dict['AngleEncoder']['ValueRefs']['U_q']

vars_rf['pid_y'] = fmu_dict['PID_Continuous']['ValueRefs']['y']
vars_rf['pid_ref'] = fmu_dict['PID_Continuous']['ValueRefs']['ref']
vars_rf['pid_u'] = fmu_dict['PID_Continuous']['ValueRefs']['u']

vars_rf['drive_u'] = fmu_dict['Drive']['ValueRefs']['u_control']
vars_rf['drive_omega'] = fmu_dict['Drive']['ValueRefs']['omega']
vars_rf['drive_torque'] = fmu_dict['Drive']['ValueRefs']['torque']

**4. Instantiate the FMUs using `FMU2Slave`**

- The FMUs are instantiated as `FMU2Slave` objects, which provide methods for initializing, simulating, and terminating the FMU.

In [9]:
from fmpy import extract
from fmpy.fmi2 import FMU2Slave

# Instantiate Reference Trajectory
unzip_dir = extract(fmu_paths['Reference'])
ref_fmu = FMU2Slave(guid=fmu_dict['Reference']["ModelDescription"].guid,
                    unzipDirectory=unzip_dir,
                    modelIdentifier=fmu_dict['Reference']["ModelDescription"].coSimulation.modelIdentifier,
                    instanceName='ref_fmu')
ref_fmu.instantiate()

# Instantiate Pendulum
unzip_dir = extract(fmu_paths['Pendulum'])
pendulum_fmu = FMU2Slave(guid=fmu_dict['Pendulum']["ModelDescription"].guid,
                          unzipDirectory=unzip_dir,
                          modelIdentifier=fmu_dict['Pendulum']["ModelDescription"].coSimulation.modelIdentifier,
                          instanceName='pendulum_fmu')
pendulum_fmu.instantiate()

# Instantiate Sensors for Reference Trajectory and Pendulum
unzip_dir = extract(fmu_paths['AngleEncoder'])
sensor_ref_fmu = FMU2Slave(guid=fmu_dict['AngleEncoder']["ModelDescription"].guid,
                         unzipDirectory=unzip_dir,
                         modelIdentifier=fmu_dict['AngleEncoder']["ModelDescription"].coSimulation.modelIdentifier,
                         instanceName='sensor_ref_fmu')
sensor_ref_fmu.instantiate()

sensor_state_fmu = FMU2Slave(guid=fmu_dict['AngleEncoder']["ModelDescription"].guid,
                         unzipDirectory=unzip_dir,
                         modelIdentifier=fmu_dict['AngleEncoder']["ModelDescription"].coSimulation.modelIdentifier,
                         instanceName='sensor_state_fmu')
sensor_state_fmu.instantiate()

# Instantiate Controller
unzip_dir = extract(fmu_paths['PID_Continuous'])
pid_fmu = FMU2Slave(guid=fmu_dict['PID_Continuous']["ModelDescription"].guid,
                    unzipDirectory=unzip_dir,
                    modelIdentifier=fmu_dict['PID_Continuous']["ModelDescription"].coSimulation.modelIdentifier,
                    instanceName='pid_fmu')
pid_fmu.instantiate()

# Instantiate Drive
unzip_dir = extract(fmu_paths['Drive'])
drive = FMU2Slave(guid=fmu_dict['Drive']["ModelDescription"].guid,
                  unzipDirectory=unzip_dir,
                  modelIdentifier=fmu_dict['Drive']["ModelDescription"].coSimulation.modelIdentifier,
                  instanceName='drive')
drive.instantiate()

fmu_list = [ref_fmu, pendulum_fmu, sensor_ref_fmu, sensor_state_fmu, pid_fmu, drive]

LOG_SOLVER        | info    | CVODE linear multistep method CV_BDF
LOG_SOLVER        | info    | CVODE maximum integration order CV_ITER_NEWTON
LOG_SOLVER        | info    | CVODE use equidistant time grid YES
LOG_SOLVER        | info    | CVODE Using relative error tolerance 1.000000e-06
LOG_SOLVER        | info    | CVODE Using dense internal linear solver SUNLinSol_Dense.
LOG_SOLVER        | info    | CVODE Use internal dense numeric jacobian method.
LOG_SOLVER        | info    | CVODE uses internal root finding method NO
LOG_SOLVER        | info    | CVODE maximum absolut step size 0
LOG_SOLVER        | info    | CVODE initial step size is set automatically
LOG_SOLVER        | info    | CVODE maximum integration order 5
LOG_SOLVER        | info    | CVODE maximum number of nonlinear convergence failures permitted during one step 10
LOG_SOLVER        | info    | CVODE BDF stability limit detection algorithm OFF
LOG_SOLVER        | info    | CVODE linear multistep method CV_BDF
LOG_S

**5. FMU Setup and Initialization**

- The FMUs are initialized by calling the `setupExperiment` method defining the start time
- The `enterInitializationMode` and `exitInitializationMode` methods are called to complete the initialization process.
- Within the initialization mode, initial values for input variables can be set using the `setReal` method.
- **Note:** We use for now the initial values and parameters defined in the Modelica models.

In [10]:
t = 0.0
tf = 10.0
h = 0.001

for fmu in fmu_list:
    #fmu.reset()
    fmu.setupExperiment(startTime=0.0)
    fmu.enterInitializationMode()
    # Set parameters
    # if fmu is ref_fmu:
        # fmu.setReal([params_rf['ref_mean'],
        #              params_rf['ref_amplitude'],
        #              params_rf['ref_frequency']],
        #             [0.0, 1.0, 0.2])
    fmu.exitInitializationMode()

# Initialize logging arrays
t_vals = []
q_ref_vals, q_state_vals, omega_state_vals = [], [], []
U_ref_vals, U_state_vals = [], []

**6. Simulation Loop**

- The simulation loop iterates over the defined time steps, advancing the simulation time by the specified step size.

**FMPy Features Used:**
- `fmu.getReal([vr])`: Retrieves the current value of a real variable identified by its value reference (VR).
- `fmu.setReal({vr: value})`: Sets the value of a real variable identified by its VR.
- `fmu.doStep(currentCommunicationPoint, communicationStepSize)`: Advances the simulation by a specified step size from the current communication point.
- `fmu.terminate()`: Terminates the FMU simulation, releasing resources.


In [11]:
while t < tf:
    # 1) Read the current reference position
    ref_fmu.doStep(t, h)
    q_ref = ref_fmu.getReal([vars_rf['q_ref']])
    
    # 2) Read the plant state at the current time
    q_state = pendulum_fmu.getReal([vars_rf['q_state']])
    omega_state = pendulum_fmu.getReal([vars_rf['omega_state']])

    # 3) Sensors: set inputs -> step -> read outputs
    sensor_ref_fmu.setReal([vars_rf['q_sensor']], q_ref)
    sensor_state_fmu.setReal([vars_rf['q_sensor']], q_state)
    sensor_ref_fmu.doStep(t, h)
    sensor_state_fmu.doStep(t, h)
    U_q_ref = sensor_ref_fmu.getReal([vars_rf['U_q']])
    U_q_state = sensor_state_fmu.getReal([vars_rf['U_q']])

    # 4) Controller: set inputs -> step -> read outputs
    pid_fmu.setReal([vars_rf['pid_y'], vars_rf['pid_ref']],
                    [U_q_state[0], U_q_ref[0]])
    pid_fmu.doStep(currentCommunicationPoint=t, communicationStepSize=h)
    u_pid = pid_fmu.getReal([vars_rf['pid_u']])

    # 5) Drive: set inputs -> step -> read outputs
    drive.setReal([vars_rf['drive_u'], vars_rf['drive_omega']],
                  [u_pid[0], omega_state[0]])
    drive.doStep(currentCommunicationPoint=t, communicationStepSize=h)
    torque = drive.getReal([vars_rf['drive_torque']])
    
    # 6) Plant: set inputs -> step
    pendulum_fmu.setReal([vars_rf['torque']], torque)
    pendulum_fmu.doStep(currentCommunicationPoint=t, communicationStepSize=h)

    # Log data
    t_vals.append(t)
    q_ref_vals.append(q_ref[0])
    q_state_vals.append(q_state[0])
    omega_state_vals.append(omega_state[0])
    U_ref_vals.append(U_q_ref[0])
    U_state_vals.append(U_q_state[0])
    
    # Advance time
    t += h

In [13]:
# for fmu in fmu_list:
#     try:
#         fmu.terminate()
#     finally:
#         fmu.freeInstance()


**7. Plotting the Results**

In [14]:
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_vals, y=q_ref_vals, mode='lines',
                            name='Reference Angle (q_ref)', line=dict(color='white', dash='dash')))
fig.add_trace(go.Scatter(x=t_vals, y=q_state_vals, mode='lines',
                            name='State Angle (q_state)', line=dict(color='red')))
fig.update_layout(
    title={'text': 'Pendulum Angle - FMUs only using FMPy', 'font': {'size': 24}},
    xaxis_title={'text': 'Time (s)', 'font': {'size': 20}},
    yaxis_title={'text': 'Angle (rad)', 'font': {'size': 20}},
    legend_title={'text': 'Legend', 'font': {'size': 18}},
    font={'size': 16},
    template='plotly_dark'
)

fig.show()

In [15]:
from SysSimX.utilities.results_opensim import create_opensim_mot_file
import numpy as np

data = {'q': np.array(q_state_vals),
        '/jointset/head_joint/q/speed': np.array(omega_state_vals)}
time = np.array(t_vals)
n_time_steps = time.shape[0]
filename = 'OpenSim/Results/demo_2_3.mot'

create_opensim_mot_file(data=data, time=time, filename=filename)